# See a Summary of All Activities

In [1]:
from IPython.display import display, Markdown
import snakemd

import fitfile
from garmindb import GarminConnectConfigManager
from garmindb.garmindb import GarminDb, Attributes, ActivitiesDb, Activities, StepsActivities, ActivityLaps, ActivityRecords
from idbutils.list_and_dict import list_not_none

from jupyter_funcs import format_number

gc_config = GarminConnectConfigManager()
db_params_dict = gc_config.get_db_params()


garmin_db = GarminDb(db_params_dict)
garmin_act_db = ActivitiesDb(db_params_dict)
measurement_system = Attributes.measurements_type(garmin_db)
unit_strings = fitfile.units.unit_strings[measurement_system]
distance_units = unit_strings[fitfile.units.UnitTypes.distance_long]

def __report_sport(sport_col, sport):
    records = Activities.row_count(garmin_act_db, sport_col, sport)
    if records > 0:
        sport_title = sport.title().replace('_', ' ')
        total_distance = Activities.get_col_sum_for_value(garmin_act_db, Activities.distance, sport_col, sport)
        if total_distance is None:
            total_distance = 0
            average_distance = 0
        else:
            average_distance = total_distance / records
        return [sport_title, records, format_number(total_distance, 1), format_number(average_distance, 1)]

doc = snakemd.new_doc()

doc.add_heading("Activities Report")
doc.add_paragraph("Analysis of all activities in the database.")

doc.add_table(
    ['Type', 'Count'],
    [
        ["Total activities", Activities.row_count(garmin_act_db)],
        ["Total Lap records", ActivityLaps.row_count(garmin_act_db)],
        ["Activity records", ActivityRecords.row_count(garmin_act_db)],
        ["Fitness activities", Activities.row_count(garmin_act_db, Activities.type, 'fitness')],
        ["Recreation activities", Activities.row_count(garmin_act_db, Activities.type, 'recreation')]
    ])

years = Activities.get_years(garmin_act_db)
years.sort()
doc.add_paragraph(f"Years with activities: {len(years)}: {years}")
sports = list_not_none(Activities.get_col_distinct(garmin_act_db, Activities.sport))
doc.add_paragraph(f"Sports: {', '.join(sports)}")
sub_sports = list_not_none(Activities.get_col_distinct(garmin_act_db, Activities.sub_sport))
doc.add_paragraph(f"SubSports: {', '.join(sub_sports)}")

sports_stats = []
for sport_name in [sport.name for sport in fitfile.Sport]:
    sport_stat = __report_sport(Activities.sport, sport_name)
    if sport_stat:
        sports_stats.append(sport_stat)
doc.add_heading("Sport Type Statistics", 3)
doc.add_table(['Sport', 'Total Activities', f'Total Distance ({distance_units})', f"Average Distance ({distance_units})"], sports_stats)

def __format_activity(activity):
    if activity:
        if activity.is_steps_activity():
            steps_activity = StepsActivities.get(garmin_act_db, activity.activity_id)
            return [activity.activity_id, activity.name, activity.type, activity.sport, format_number(activity.distance, 1), activity.elapsed_time, format_number(activity.avg_speed, 1),
                    steps_activity.avg_pace, format_number(activity.calories), format_number(activity.training_load, 1), activity.self_eval_feel, activity.self_eval_effort]
        return [activity.activity_id, activity.name, activity.type, activity.sport, format_number(activity.distance, 1), activity.elapsed_time, format_number(activity.avg_speed, 1), '',
                format_number(activity.calories), format_number(activity.training_load, 1), activity.self_eval_feel, activity.self_eval_effort]
    return ['', '', '', '', '', '', '', '', '']

activities = Activities.get_latest(garmin_act_db, 10)
rows = [__format_activity(activity) for activity in activities]
doc.add_heading("Last Ten Activities", 3)
doc.add_table(['Id', 'Name', 'Type', 'Sport', f'Distance ({distance_units})', 'Elapsed Time', f'Speed ({unit_strings[fitfile.units.UnitTypes.speed]})',
               f'Pace ({unit_strings[fitfile.units.UnitTypes.pace]})', 'Calories', 'Training Load', 'Feel', 'Effort'], rows)

rows = []
for display_activity in gc_config.display_activities():
    name = display_activity.activity_name().capitalize()
    rows.append([f'Latest {name}'] + __format_activity(Activities.get_latest_by_sport(garmin_act_db, display_activity)))
    rows.append([f'Fastest {name}'] + __format_activity(Activities.get_fastest_by_sport(garmin_act_db, display_activity)))
    rows.append([f'Slowest {name}'] + __format_activity(Activities.get_slowest_by_sport(garmin_act_db, display_activity)))
    rows.append([f'Longest {name}'] + __format_activity(Activities.get_longest_by_sport(garmin_act_db, display_activity)))

doc.add_heading("Interesting Activities", 3)
doc.add_table(['What', 'Id', 'Name', 'Type', 'Sport', f'Distance ({distance_units})', 'Elapsed Time', f'Speed ({unit_strings[fitfile.units.UnitTypes.speed]})',
               f'Pace ({unit_strings[fitfile.units.UnitTypes.pace]})', 'Calories', 'Training Load', 'Feel', 'Effort'], rows)

doc.add_heading("Courses", 3)
courses = Activities.get_col_distinct(garmin_act_db, Activities.course_id)
doc.add_paragraph(str(courses))

display(Markdown(str(doc)))

# Activities Report

Analysis of all activities in the database.

| Type                  | Count   |
| --------------------- | ------- |
| Total activities      | 5284    |
| Total Lap records     | 7598    |
| Activity records      | 2798491 |
| Fitness activities    | 27      |
| Recreation activities | 1       |

Years with activities: 4: [2022, 2023, 2024, 2025]

Sports: walking, generic, running, fitness_equipment, training, other, cycling, Other, swimming, transition, UnknownEnumValue_67

SubSports: generic, cardio_training, strength_training, exercise, pilates, yoga, breathing, treadmill, casual_walking, yoga_gym, indoor_cycling, track, street, lap_swimming, open_water, flexibility_training

### Sport Type Statistics

| Sport             | Total Activities | Total Distance (kilometers) | Average Distance (kilometers) |
| ----------------- | ---------------- | --------------------------- | ----------------------------- |
| Generic           | 2670             | 1.3                         | 0.0                           |
| Running           | 451              | 1019.8                      | 2.3                           |
| Cycling           | 6                | 3.2                         | 0.5                           |
| Transition        | 3                | 0.0                         | 0.0                           |
| Fitness Equipment | 303              | 1.3                         | 0.0                           |
| Swimming          | 36               | 6.4                         | 0.2                           |
| Training          | 804              | 0.0                         | 0.0                           |
| Walking           | 843              | 728.1                       | 0.9                           |

### Last Ten Activities

| Id          | Name                        | Type          | Sport               | Distance (kilometers) | Elapsed Time    | Speed (kph) | Pace (per kilometers) | Calories | Training Load | Feel   | Effort |
| ----------- | --------------------------- | ------------- | ------------------- | --------------------- | --------------- | ----------- | --------------------- | -------- | ------------- | ------ | ------ |
| 20885987515 | None                        | None          | UnknownEnumValue_67 | -                     | 00:10:00.100000 | -           |                       | -        | -             | None   | None   |
| 20885907462 | Drapetsona Running Day#1315 | uncategorized | running             | 2.7                   | 00:31:31        | 5.2         | 00:18:43.059487       | 210      | -             | Strong | Hard   |
| 20885348882 | None                        | None          | training            | -                     | 00:12:12.403000 | -           |                       | -        | -             | None   | None   |
| 20885348669 | F3b Monitor+HRV             | uncategorized | generic             | 0.0                   | 00:03:20.881000 | 0.0         |                       | 4        | -             | None   | None   |
| 20885348551 | HRV                         | uncategorized | generic             | 0.0                   | 00:05:00.266000 | 0.0         |                       | 8        | -             | None   | None   |
| 20885348306 | HRV                         | uncategorized | generic             | 0.0                   | 00:04:59.058000 | 0.0         |                       | 9        | -             | None   | None   |
| 20882962937 | F3b Monitor+HRV             | uncategorized | generic             | 0.0                   | 00:03:36.360000 | 0.0         |                       | 9        | -             | None   | None   |
| 20879108848 | F3b Monitor+HRV             | uncategorized | generic             | 0.0                   | 00:02:24.308000 | 0.0         |                       | 6        | -             | None   | None   |
| 20875476947 | HRV                         | uncategorized | generic             | 0.0                   | 00:05:00.268000 | 0.0         |                       | 7        | -             | None   | None   |
| 20875476614 | F3b Monitor+HRV             | uncategorized | generic             | 0.0                   | 00:02:16.578000 | 0.0         |                       | 3        | -             | None   | None   |

### Interesting Activities

| What         | Id          | Name                        | Type          | Sport   | Distance (kilometers) | Elapsed Time    | Speed (kph) | Pace (per kilometers) | Calories | Training Load | Feel   | Effort     |
| ------------ | ----------- | --------------------------- | ------------- | ------- | --------------------- | --------------- | ----------- | --------------------- | -------- | ------------- | ------ | ---------- |
| Latest Walk  | 20865193634 | Piraeus Walking             | uncategorized | walking | 0.5                   | 00:09:01.583000 | 4.3         | 00:22:36.951285       | 43       | -             | None   | None       |
| Fastest Walk | 17997231529 | Glyfada Walking             | uncategorized | walking | 0.4                   | 00:03:27.307000 | 7.9         | 00:12:13.186451       | 20       | -             | None   | None       |
| Slowest Walk | 11325508010 | Piraeus Other               | uncategorized | walking | 2.0                   | 01:24:55        | 1.4         | 01:08:24.113695       | 0        | -             | None   | None       |
| Longest Walk | 11178515849 | Athens Walking              | uncategorized | walking | 5.9                   | 02:01:22        | 2.9         | 00:33:06.844698       | 459      | -             | None   | None       |
| Latest Run   | 20885907462 | Drapetsona Running Day#1315 | uncategorized | running | 2.7                   | 00:31:31        | 5.2         | 00:18:43.059487       | 210      | -             | Strong | Hard       |
| Fastest Run  | 11178515838 | Running                     | uncategorized | running | 0.0                   | 00:00:01        | 60.0        | 00:01:36.545124       | 144      | -             | None   | None       |
| Slowest Run  | 17345590182 | Running                     | uncategorized | running | 0.0                   | 00:00:04.446000 | 0.0         | 00:00:00              | 0        | -             | Normal | Very Light |
| Longest Run  | 20231261521 | Drapetsona Running          | uncategorized | running | 7.0                   | 01:26:49        | 4.9         | 00:19:52.990539       | 568      | -             | Normal | Very Hard  |
| Latest Ride  | 11178516031 | Cycling                     | uncategorized | cycling | 0.0                   | 00:11:20        | 0.0         |                       | 55       | -             | None   | None       |
| Fastest Ride | 11178515844 | Indoor Cycling              | uncategorized | cycling | 2.4                   | 00:08:00        | 18.0        |                       | 31       | -             | None   | None       |
| Slowest Ride | 11178516031 | Cycling                     | uncategorized | cycling | 0.0                   | 00:11:20        | 0.0         |                       | 55       | -             | None   | None       |
| Longest Ride | 11178515844 | Indoor Cycling              | uncategorized | cycling | 2.4                   | 00:08:00        | 18.0        |                       | 31       | -             | None   | None       |

### Courses

[None, 179203070, 280598925]